# Total Checkup EDA

`adoc_v1.total_checkups.json` 데이터에 대한 탐색적 데이터 분석(EDA)입니다.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from core.loader import DataLoader
from core.eda import CheckupDateAnalyzer

OUTPUT_DIR = Path("../outputs/260522_EDA")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. 검진일 분포 (Checkup Date Distribution)

전체 검진 건수·인원·기간 개요 및 연도/월별 분포를 확인합니다.

In [2]:
EXCLUDE_USER_KEYS = [
    "INVALID_RESULT",  # user_key가 유효하지 않은 이상치 (32회 검진)
]

df = DataLoader("adoc_v1.total_checkups.json", exclude_user_keys=EXCLUDE_USER_KEYS).load()
analyzer = CheckupDateAnalyzer(df, date_col="checkup_date")

In [3]:
summary = analyzer.summary()
print(f"검진 건수 : {summary['count']:,}건")
print(f"검진 인원 : {summary['user_count']:,}명")
print(f"최초 검진일: {summary['min'].strftime('%Y-%m-%d')}")
print(f"최근 검진일: {summary['max'].strftime('%Y-%m-%d')}")
print(f"포함 연도  : {summary['unique_years']}")

검진 건수 : 35,833건
검진 인원 : 29,597명
최초 검진일: 2019-09-10
최근 검진일: 2026-02-28
포함 연도  : [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [4]:
analyzer.plot_distribution(output_dir=OUTPUT_DIR, show=False)

/data2/mason/prediabetes_diabetes/core/eda.py:78: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  month_counts = dates.dt.to_period("M").value_counts().sort_index()


Saved: ../outputs/260522_EDA/checkup_date_distribution.png


## 2. 반복 검진 분석 (Repeated Checkup Analysis)

두 번 이상 검진을 받은 환자 현황과 재방문 주기 분포를 분석합니다.

In [5]:
from core.interval import CheckupIntervalAnalyzer

interval_analyzer = CheckupIntervalAnalyzer(df, date_col="checkup_date", user_col="user_key")

In [6]:
s = interval_analyzer.summary()
print(f"전체 환자 수       : {s['total_patients']:,}명")
print(f"1회만 검진         : {s['single_visit']:,}명")
print(f"2회 이상 검진      : {s['repeat_visit']:,}명  ({s['repeat_ratio']}%)")
print(f"최다 검진 횟수     : {s['max_visits']}회")
print()
print(f"재방문 주기 중앙값 : {s['median_interval_days']}일 ({s['median_interval_days']/30.44:.1f}개월)")
print(f"재방문 주기 평균   : {s['mean_interval_days']}일 ({s['mean_interval_days']/30.44:.1f}개월)")
print(f"IQR               : {s['interval_q1']}일 ~ {s['interval_q3']}일")

전체 환자 수       : 29,597명
1회만 검진         : 24,811명
2회 이상 검진      : 4,786명  (16.2%)
최다 검진 횟수     : 5회

재방문 주기 중앙값 : 371.0일 (12.2개월)
재방문 주기 평균   : 439.2일 (14.4개월)
IQR               : 336.0일 ~ 448.0일


In [7]:
interval_analyzer.plot_analysis(output_dir=OUTPUT_DIR, show=False)

Saved: ../outputs/260522_EDA/repeated_checkup_analysis.png


## 3. 결측치 분석 (Missing Value Analysis)

2회 이상 검진을 받은 수검자를 대상으로 공복혈당·당화혈색소 결측 현황을 확인합니다.

In [8]:
from core.missing import MissingValueAnalyzer

missing_analyzer = MissingValueAnalyzer(df, min_visits=2)

In [9]:
missing_analyzer.summary()

,field_code,field_name,total_records,present_records,missing_records,missing_rate_pct
0,CH164,공복혈당,11039,10804,235,2.1
1,CH161,당화혈색소,11039,4855,6184,56.0


In [10]:
missing_analyzer.plot_missing(output_dir=OUTPUT_DIR, show=False)

Saved: ../outputs/260522_EDA/missing_value_analysis.png


## 4. 당뇨 상태 전이 분석 (Glucose State Transition)

공복혈당 기준으로 상태를 분류하고, 2회 이상 수검자의 검진 간 전이 흐름을 분석합니다.

| 상태 | 공복혈당 기준 |
|------|--------------|
| 정상 | 100 미만 |
| 당뇨병전단계 | 100 이상 125 이하 |
| 당뇨병 | 125 초과 |

In [11]:
from core.transition import GlucoseTransitionAnalyzer

transition_analyzer = GlucoseTransitionAnalyzer(df, min_visits=2)

In [12]:
import pandas as pd
pd.set_option("display.max_rows", None)

print("=== 전체 전이 요약 ===")
display(transition_analyzer.summary())

seq_dict = transition_analyzer.sequence_summary(n_visits=[2, 3, 4, 5])
for n, df in seq_dict.items():
    sep = "=" * 30
    print(sep)
    print(f"{n}회 수검자 시퀀스")
    print(sep)
    display(df)


=== 전체 전이 요약 ===


,from_state,to_state,count,label,pct
0,정상,정상,3293,정상 → 정상,54.6
1,당뇨병전단계,당뇨병전단계,1006,당뇨병전단계 → 당뇨병전단계,16.7
2,정상,당뇨병전단계,742,정상 → 당뇨병전단계,12.3
3,당뇨병전단계,정상,651,당뇨병전단계 → 정상,10.8
4,당뇨병,당뇨병,159,당뇨병 → 당뇨병,2.6
5,당뇨병전단계,당뇨병,82,당뇨병전단계 → 당뇨병,1.4
6,당뇨병,당뇨병전단계,70,당뇨병 → 당뇨병전단계,1.2
7,당뇨병,정상,14,당뇨병 → 정상,0.2
8,정상,당뇨병,10,정상 → 당뇨병,0.2


2회 수검자 시퀀스


,sequence,count,pct
0,정상 → 정상,1908,55.7
1,당뇨병전단계 → 당뇨병전단계,562,16.4
2,정상 → 당뇨병전단계,405,11.8
3,당뇨병전단계 → 정상,369,10.8
4,당뇨병 → 당뇨병,91,2.7
5,당뇨병전단계 → 당뇨병,49,1.4
6,당뇨병 → 당뇨병전단계,35,1.0
7,당뇨병 → 정상,6,0.2
8,정상 → 당뇨병,3,0.1


3회 수검자 시퀀스


,sequence,count,pct
0,정상 → 정상 → 정상,490,48.7
1,당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계,101,10.0
2,정상 → 당뇨병전단계 → 정상,88,8.7
3,정상 → 정상 → 당뇨병전단계,78,7.8
4,당뇨병전단계 → 당뇨병전단계 → 정상,49,4.9
5,정상 → 당뇨병전단계 → 당뇨병전단계,48,4.8
6,당뇨병전단계 → 정상 → 정상,41,4.1
7,당뇨병전단계 → 정상 → 당뇨병전단계,41,4.1
8,당뇨병 → 당뇨병 → 당뇨병,21,2.1
9,당뇨병전단계 → 당뇨병전단계 → 당뇨병,8,0.8


4회 수검자 시퀀스


,sequence,count,pct
0,정상 → 정상 → 정상 → 정상,72,37.3
1,당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계,19,9.8
2,정상 → 당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계,12,6.2
3,정상 → 정상 → 정상 → 당뇨병전단계,12,6.2
4,정상 → 당뇨병전단계 → 정상 → 정상,12,6.2
5,정상 → 정상 → 당뇨병전단계 → 정상,9,4.7
6,정상 → 정상 → 당뇨병전단계 → 당뇨병전단계,7,3.6
7,정상 → 당뇨병전단계 → 정상 → 당뇨병전단계,5,2.6
8,당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계 → 정상,5,2.6
9,당뇨병전단계 → 정상 → 당뇨병전단계 → 당뇨병전단계,4,2.1


5회 수검자 시퀀스


,sequence,count,pct
0,당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계,1,50.0
1,당뇨병 → 당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계,1,50.0


In [13]:
transition_analyzer.plot_sequences(n_visits=[2, 3, 4, 5], top_n=10, output_dir=OUTPUT_DIR, show=False)

Saved: ../outputs/260522_EDA/glucose_sequences.png


## 5. 데이터셋 생성 (Dataset Export)

2회 수검자 중 **첫 검진이 정상**인 수검자를 아래 기준으로 분류하여 JSON으로 저장합니다.

| dataset | 전이 패턴 |
|---------|----------|
| `pre-diabetes` | 정상 → 정상 |
| `diabetes` | 정상 → 당뇨병전단계 / 정상 → 당뇨병 |

In [14]:
dataset_df = transition_analyzer.export_dataset(
    output_path=OUTPUT_DIR / "two_visit_dataset.json"
)

for ds in ["pre-diabetes", "diabetes"]:
    sub = dataset_df[dataset_df["dataset"] == ds]
    print(f"[{ds}] 총 {len(sub):,}명")
    print(sub["label"].value_counts().rename({0: "label=0", 1: "label=1"}).to_string())
    print()

Saved: ../outputs/260522_EDA/two_visit_dataset.json  (3,292명)
[pre-diabetes] 총 2,313명
label
label=0    1905
label=1     408

[diabetes] 총 979명
label
label=0    930
label=1     49



### 3회 이상 수검자 데이터셋

**분류 기준**

| dataset | label | 조건 |
|---------|-------|------|
| pre-diabetes | 0 | 정상→정상 (직전 상태가 전단계/당뇨 아닌 경우) |
| pre-diabetes | 1 | 정상→전단계/당뇨 (직전 상태가 전단계/당뇨 아닌 경우) |
| diabetes | 0 | 전단계→전단계/정상 (직전 상태가 당뇨 아닌 경우) |
| diabetes | 1 | 전단계→당뇨 (직전 상태가 당뇨 아닌 경우) |

**중복 제거 규칙** (동일 dataset 내 user_key는 반드시 유일)

- 동일 user_key에 유효한 전이 흐름이 여러 개인 경우
  - label=0 후보가 여럿: 이후 검진일의 공복혈당이 **가장 낮은** 검진일 선택
  - label=1 후보가 여럿: 이후 검진일의 공복혈당이 **가장 높은** 검진일 선택
- label=0 과 label=1 후보가 동시에 존재하는 경우: **label=1 우선 보존**

In [15]:
multi_result = transition_analyzer.export_multi_visit_dataset(
    min_visits=3,
    output_path=OUTPUT_DIR / "multi_visit_dataset.json",
)

[pre-diabetes] 총 849명  (label=0: 566, label=1: 283)
[diabetes] 총 494명  (label=0: 469, label=1: 25)
Saved: ../outputs/260522_EDA/multi_visit_dataset.json


## 6. 검진 간격 분포 분석 (Checkup Interval Distribution)

`two_visit_dataset.json` 과 `multi_visit_dataset.json` 을 종합하여
`current_checkup_date` → `future_checkup_date` 사이의 일수 분포를 dataset × label × 방문 횟수 기준으로 시각화합니다.

In [16]:
from core.dataset_analyzer import DatasetIntervalAnalyzer

interval_analyzer = DatasetIntervalAnalyzer(
    OUTPUT_DIR / "two_visit_dataset.json",
    OUTPUT_DIR / "multi_visit_dataset.json",
)

In [17]:
interval_analyzer.summary()

,dataset,label,source,count,mean_days,median_days,q1_days,q3_days,min_days,max_days
0,diabetes,0,2회,930,446.5,374.0,338.0,477.5,0,1492
1,diabetes,0,3회+,469,379.2,364.0,329.0,395.0,121,1003
2,diabetes,1,2회,49,509.8,405.0,344.0,698.0,219,1155
3,diabetes,1,3회+,25,415.6,383.0,360.0,463.0,221,732
4,pre-diabetes,0,2회,1905,486.8,389.0,343.0,645.0,0,1595
5,pre-diabetes,0,3회+,566,392.2,367.0,336.0,405.8,0,1148
6,pre-diabetes,1,2회,408,486.2,383.0,343.0,659.0,0,1497
7,pre-diabetes,1,3회+,283,392.1,369.0,329.0,400.0,193,869


In [18]:
interval_analyzer.plot_interval_distribution(output_dir=OUTPUT_DIR, show=False)

Saved: ../outputs/260522_EDA/interval_dist_by_dataset_label.png
